# YOLO11n training and evaluation

Reproduce the 640/960 experiment with Aquarium Combined v6. Download the dataset separately; select models on validation before evaluating test.


In [ ]:
%pip install "ultralytics==8.4.115" "PyYAML>=6.0,<7.0"

In [ ]:
from pathlib import Path
import subprocess
import yaml
from ultralytics import YOLO

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists():
    if (REPO_ROOT.parent / "configs").exists():
        REPO_ROOT = REPO_ROOT.parent
    else:
        REPO_ROOT = Path.cwd() / "aquarium-yolo11-portfolio"
        if not REPO_ROOT.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/CpfPatrick/aquarium-yolo11-portfolio.git",
                str(REPO_ROOT)], check=True)
BASELINE_CONFIG = REPO_ROOT / "configs" / "baseline.yaml"
EXPERIMENT_CONFIG = REPO_ROOT / "configs" / "exp01_imgsz960.yaml"


In [ ]:
# Download Aquarium Combined v6 in YOLO format, then set this path.
DATA_YAML = REPO_ROOT / "data" / "aquarium-combined-v6-yolov11" / "data.yaml"
print("Dataset available:", DATA_YAML.is_file())


In [ ]:
def load_portable_config(path: Path, data_yaml: Path) -> dict:
    config = yaml.safe_load(path.read_text(encoding='utf-8'))
    config['data'] = str(data_yaml.resolve())
    config['project'] = str((REPO_ROOT / 'runs').resolve())
    return config

baseline = load_portable_config(BASELINE_CONFIG, DATA_YAML)
experiment = load_portable_config(EXPERIMENT_CONFIG, DATA_YAML)
changed = {key for key in set(baseline) | set(experiment) if baseline.get(key) != experiment.get(key)}
assert changed == {'imgsz', 'name'}, changed
print('Registered differences:', sorted(changed))

In [ ]:
def run_training(config_path: Path, data_yaml: Path):
    if not data_yaml.is_file():
        raise FileNotFoundError(f"Download the dataset and set DATA_YAML: {data_yaml}")
    config = load_portable_config(config_path, data_yaml)
    weights = config.pop('model')
    config.pop('task', None)
    config.pop('mode', None)
    return YOLO(weights).train(**config)

In [ ]:
def run_evaluation(
    checkpoint: Path,
    data_yaml: Path,
    imgsz: int,
    split: str = 'val',
    approved_test: bool = False,
):
    if not data_yaml.is_file():
        raise FileNotFoundError(f"Download the dataset and set DATA_YAML: {data_yaml}")
    if split == 'test' and not approved_test:
        raise PermissionError('Frozen test requires a checkpoint selected using validation only.')
    if split not in {'val', 'test'}:
        raise ValueError('split must be val or test')
    return YOLO(str(checkpoint)).val(
        data=str(data_yaml),
        split=split,
        imgsz=imgsz,
        batch=8,
        device=0,
        conf=0.001,
        iou=0.7,
        rect=True,
        plots=True,
        workers=2,
    )

## Run
Set the dataset path, then run either training call below. Test is evaluated only after model selection.

```python
baseline_run = run_training(BASELINE_CONFIG, DATA_YAML)
experiment_run = run_training(EXPERIMENT_CONFIG, DATA_YAML)
run_evaluation(REPO_ROOT / "runs/yolo11n_imgsz960_seed42/weights/best.pt",
               DATA_YAML, imgsz=960, split="val")
# After selection, use split="test", approved_test=True.
```

Published metrics are in `results/final_test/`; reruns can vary across hardware and dependency versions.
